# GHEREMIAH AI

This is a learning course for creating a small LLM from scratch. We will follow the 8 steps of creating ML models:
1. Problem Definition
2. Data Collection
3. Data Preprocessing & Cleaning
4. Exploratory Data Analysis (EDA)
5. Feature Engineering
6. Model Selection
7. Training & Evaluation
8. Deployment & Monitoring

## 1. Problem Definition

We want to build a small autoregressive language model that can generate text, answer chat-like prompts, and handle simple tool-use behavior in a controlled setting.

### Problem Statement
Create a compact decoder-only Transformer model from scratch that learns to predict the next token in a sequence. The model should be able to:
- generate fluent and coherent English text,
- respond in a chat-like style to short prompts,
- solve simple reasoning tasks such as basic arithmetic, common-sense questions, or short multi-step logic,
- and recognize when a prompt suggests a tool call and format a simple structured action request.

### Inputs and Outputs
- Input: a text prompt, conversation history, or task description.
- Output: the next tokens of a continuation, a helpful reply, or a structured tool-call request when appropriate.

### Functional Requirements
- The model should produce readable, grammatically reasonable text.
- It should follow instructions and stay consistent within a short context window.
- It should learn basic patterns of reasoning and tool-use behavior from synthetic or curated training data.
- It should remain small enough to train and run on a single GPU in a Colab-style environment.

### Non-Goals
- We are not trying to build a state-of-the-art general-purpose assistant.
- We are not aiming for perfect factual accuracy, long-term memory, or complex planning.
- We are not trying to connect to real external tools yet; this first version focuses on learning the pattern of tool invocation.

### Constraints
- Model size should stay small (for example, 1–8 layers and 128–512 hidden dimensions).
- Training should be feasible on modest hardware.
- The dataset should be simple, structured, and easy to understand for learning purposes.

### Success Criteria
A first version is considered successful if:
- the model produces coherent text samples after training,
- it can answer simple prompts in a natural way,
- it can complete basic reasoning-style examples with reasonable accuracy,
- and it can follow simple tool-call formatting examples when provided in the training data.

### Deliverable
A working small language model with a tokenizer, training code, and an inference script that can generate text from a prompt.

### Later Additions
- Improve reasoning quality by training on larger or more diverse data.
- Optimize for cross-platform deployment, including export to ONNX.
- Extend the model to support more complex tool calling, code understanding, and multi-step agent behavior.

## Installing Needed Resources and Packages

In [1]:
# Global environment switch: keep this fixed for the whole notebook lifecycle.
USE_COLAB = True

# Install packages
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install datasets tiktoken wandb tqdm numpy

import os
import shutil
import torch
from pathlib import Path

# Storage layout
if USE_COLAB:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        PROJECT_ROOT = Path('/content')
        DRIVE_PROJECT_ROOT = Path('/content/drive/MyDrive/GheremiahAI')
        DRIVE_PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
        RAW_ROOT = PROJECT_ROOT / 'raw_dataset'
        DATASET_ROOT = PROJECT_ROOT / 'dataset'
        CHECKPOINT_ROOT = DRIVE_PROJECT_ROOT / 'checkpoints'
    except Exception as exc:
        print('Colab Drive mount failed, switching to local fallback:', exc)
        USE_COLAB = False
        PROJECT_ROOT = Path.cwd()
        DRIVE_PROJECT_ROOT = PROJECT_ROOT
        RAW_ROOT = PROJECT_ROOT / 'raw_dataset'
        DATASET_ROOT = PROJECT_ROOT / 'dataset'
        CHECKPOINT_ROOT = PROJECT_ROOT / 'checkpoints'
else:
    PROJECT_ROOT = Path.cwd()
    DRIVE_PROJECT_ROOT = PROJECT_ROOT
    RAW_ROOT = PROJECT_ROOT / 'raw_dataset'
    DATASET_ROOT = PROJECT_ROOT / 'dataset'
    CHECKPOINT_ROOT = PROJECT_ROOT / 'checkpoints'

RAW_ROOT.mkdir(parents=True, exist_ok=True)
DATASET_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)

print('USE_COLAB =', USE_COLAB)
print('PROJECT_ROOT =', PROJECT_ROOT)
print('RAW_ROOT =', RAW_ROOT)
print('DATASET_ROOT =', DATASET_ROOT)
print('CHECKPOINT_ROOT =', CHECKPOINT_ROOT)

print('CUDA available:', torch.cuda.is_available())
print('CUDA device count:', torch.cuda.device_count())

if torch.cuda.is_available():
    device = torch.device('cuda')
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    print('Using GPU:', torch.cuda.get_device_name(0))
else:
    device = torch.device('cpu')
    print('Using CPU fallback')

Looking in indexes: https://download.pytorch.org/whl/cu121
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
USE_COLAB = True
PROJECT_ROOT = /content
RAW_ROOT = /content/raw_dataset
DATASET_ROOT = /content/dataset
CHECKPOINT_ROOT = /content/drive/MyDrive/GheremiahAI/checkpoints
CUDA available: True
CUDA device count: 1
Using GPU: Tesla T4


In [2]:
import os

os.chdir(PROJECT_ROOT)
print('Working directory:', Path.cwd())
print('Drive persistence root:', DRIVE_PROJECT_ROOT)


Working directory: /content
Drive persistence root: /content/drive/MyDrive/GheremiahAI


## 2. Data Collection

### Overview
For pre-training a small LLM from scratch, we need a large corpus of high-quality, coherent text. Real web-scale data (e.g., CommonCrawl) is too noisy and computationally expensive for learning/experimentation. Instead, we use curated or synthetic datasets optimized for small models.

We prioritize datasets that:
- Contain diverse, well-formed English text.
- Are small enough for rapid iteration on Colab.
- Enable quick observation of learning progress (e.g., coherent story generation).

### Datasets Used

#### Primary Dataset: TinyStories
- **Description**: A collection of short synthetic stories generated by GPT-3.5 and GPT-4. Stories use simple vocabulary (suitable for 3–4 year olds) but maintain coherent plots, grammar, and reasoning. This dataset was specifically designed to study how small language models can still produce fluent, consistent text.
- **Size**: ~2–4 million stories (train split provides hundreds of millions of tokens depending on preprocessing). The full `TinyStories_all_data.tar.gz` is ~1.6 GB compressed.
- **Source / URL**:
  - Hugging Face Dataset: [https://huggingface.co/datasets/roneneldan/TinyStories](https://huggingface.co/datasets/roneneldan/TinyStories)
  - Direct download: [https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStories_all_data.tar.gz](https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStories_all_data.tar.gz)
  - Paper: [TinyStories: How Small Can Language Models Be and Still Speak Coherent English?](https://arxiv.org/abs/2305.07759)
- **Format**: 
  - Raw: `.txt` files or `.tar.gz` archive containing JSONL-like stories (one story per line or structured JSON).
  - Preprocessed: Converted to binary `.bin` files (packed token IDs as `uint16` or `int32`) or used directly via `datasets` library as text.
- **Why this format?**
  - **.txt / JSONL**: Human-readable, easy to inspect, parse, and clean. Ideal for initial EDA and custom tokenization.
  - **Binary `.bin`**: Extremely efficient for training—fast loading, low memory overhead during batching. Common in from-scratch implementations (e.g., nanoGPT style) to avoid repeated text decoding.
  - Supports streaming/large-scale processing without loading everything into RAM.

#### Secondary / Alternative Dataset: Tiny Shakespeare (for initial experiments)
- **Description**: Complete works of William Shakespeare (~1 MB raw text). Classic benchmark for character-level language models.
- **Source / URL**: Often included in nanoGPT repo examples, or downloadable from public sources like [https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt](https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt) (or via Hugging Face).
- **Format**: Plain `.txt` file (raw text).
- **Why this format?**: Extremely simple for character-level tokenization (vocab ~65 chars). Perfect for debugging the full pipeline before moving to subword tokenization. Low barrier for quick experiments.

### Data Collection Process
1. **Access**: Use `datasets` library in Colab for Hugging Face datasets or `wget`/`requests` for direct downloads.
2. **License Check**: TinyStories uses CDLA-Sharing-1.0 (permissive for research/education). Verify before use.
3. **Volume**: Start with a subset (e.g., first 10–50k stories) for hyperparameter tuning, then scale to full dataset.
4. **Ethical Considerations**: Synthetic data reduces privacy/bias risks from real user text. Still, monitor for any inherited biases from the generating models.
5. **Storage**: Mount Google Drive in Colab to persist raw and processed files across sessions.

### Rationale for Dataset Choices
- **TinyStories**: Enables small models to achieve impressive coherence (as shown in the original paper). Avoids the noise of web data while still teaching real language structure. Ideal bridge between toy datasets (Shakespeare) and real pre-training.
- **Shakespeare**: Fastest way to validate the entire pipeline (tokenization → training → generation) in minutes.
- **Why not larger web datasets?** They require massive compute, heavy cleaning, and longer training—counter to our educational "from scratch" goal on Colab.

## Downloading the datasets
First, we will download tiny stories from huggingface, extract it and save it in raw datasets

In [3]:
# Local runtime directory for downloads/extraction
raw_dir = RAW_ROOT
raw_dir.mkdir(parents=True, exist_ok=True)

print(f'Raw dataset directory created at: {raw_dir}')

Raw dataset directory created at: /content/raw_dataset


In [16]:
# DOWNLOADING TINY STORIES

import requests
from tqdm import tqdm

url = "https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStories_all_data.tar.gz"
filename = raw_dir / "TinyStories_all_data.tar.gz"

print("Downloading TinyStories_all_data.tar.gz (~1.6 GB compressed)... This may take a while.")

response = requests.get(url, stream=True)
total_size = int(response.headers.get('content-length', 0))

with open(filename, 'wb') as f, tqdm(
    desc=filename.name,
    total=total_size,
    unit='iB',
    unit_scale=True,
    unit_divisor=1024,
) as bar:
    for chunk in response.iter_content(chunk_size=8192):
        size = f.write(chunk)
        bar.update(size)

print(f"\nDownload complete! File saved to: {filename}")

TinyStories_all_data.tar.gz: 100%|██████████| 1.50G/1.50G [00:25<00:00, 64.1MiB/s]   


Download complete! File saved to: /content/raw_dataset/TinyStories_all_data.tar.gz


In [18]:
# EXTRACT TINY MODELS

import tarfile

tar_path = raw_dir / "TinyStories_all_data.tar.gz"
extract_dir = raw_dir / "tinystories_raw"

print("Extracting TinyStories... This can take several minutes.")

with tarfile.open(tar_path, 'r:gz') as tar:
    tar.extractall(path=extract_dir)

print(f"Extraction complete! Files are in: {extract_dir}")

drive_extract_dir = DRIVE_PROJECT_ROOT / 'tinystories_raw'

# Check if the extraction was successful
if extract_dir.exists():
    print('Copied extracted dataset snapshot to Drive:', drive_extract_dir)

    if USE_COLAB:        
        shutil.copytree(extract_dir, drive_extract_dir)
else:
    print("Extraction failed. Directory does not exist.")

Extracting TinyStories... This can take several minutes.


/tmp/ipykernel_15541/3181675984.py:11: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=extract_dir)


Extraction complete! Files are in: /content/raw_dataset/tinystories_raw
Copied extracted dataset snapshot to Drive: /content/drive/MyDrive/GheremiahAI/tinystories_raw


In [19]:
# VERIFY EXTRACTED FILES

# List the extracted files
extracted_files = list(extract_dir.glob("**/*"))
print(f"Total extracted items: {len(extracted_files)}")

# Show first few files (stories are usually .txt or JSON-like)
for i, file in enumerate(extracted_files[:10]):
    print(f"{i+1}: {file.name} ({file.stat().st_size / (1024*1024):.2f} MB)")

Total extracted items: 50
1: data01.json (133.87 MB)
2: data26.json (134.02 MB)
3: data24.json (133.82 MB)
4: data03.json (133.88 MB)
5: data05.json (133.98 MB)
6: data37.json (133.79 MB)
7: data29.json (134.07 MB)
8: data09.json (133.86 MB)
9: data12.json (133.86 MB)
10: data49.json (90.82 MB)


In [9]:
# DELETING THE TAR.GZ FILE TO SAVE STORAGE AFTER ITS EXTRACTION

if tar_path.exists():
    tar_path.unlink()
    print("Compressed .tar.gz deleted to save space.")

Compressed .tar.gz deleted to save space.


Downloading TinyShakespeare dataset

In [20]:
# Download the actual Shakespeare corpus into a real text file.
# Use -O for output content, not -o, which writes the download log.
!wget -q -O /content/raw_dataset/TinyShakespeare.txt https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
print('Downloaded TinyShakespeare to:', Path('/content/raw_dataset/TinyShakespeare.txt'))

Downloaded TinyShakespeare to: /content/raw_dataset/TinyShakespeare.txt


## 3. Data Preprocessing & Cleaning

We now turn the downloaded raw text into a consistent training corpus. For this notebook, we will:
1. collect text from the raw dataset files,
2. normalize formatting and remove unusable records,
3. create a clean text set for tokenization,
4. add a few synthetic reasoning and tool-call demonstrations to anchor the learning objective.


In [21]:
from pathlib import Path
import re
import json

raw_root = RAW_ROOT
tiny_dir = raw_root / 'tinystories_raw'
shakespeare_path = raw_root / 'TinyShakespeare.txt'


def normalize_text(text: str) -> str:
    text = text.replace('\r', '\n')
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'[ \t]{2,}', ' ', text)
    text = re.sub(r'(?<=[A-Za-z])\n(?=[A-Za-z])', ' ', text)
    return text.strip()


def add_record(text: str):
    text = normalize_text(text)
    if 30 <= len(text) <= 900:
        stories.append(text)


stories = []
for file_path in tiny_dir.rglob('*'):
    if file_path.suffix.lower() not in {'.txt', '.json', '.jsonl'}:
        continue

    try:
        content = file_path.read_text(encoding='utf-8', errors='ignore')
    except Exception:
        continue

    if file_path.suffix.lower() in {'.json', '.jsonl'}:
        for line in content.splitlines():
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                obj = None

            if isinstance(obj, dict):
                candidate = None
                for key in ('story', 'text', 'content', 'prompt', 'output'):
                    if isinstance(obj.get(key), str):
                        candidate = obj[key]
                        break
                if candidate is not None:
                    add_record(candidate)
                else:
                    add_record(str(obj))
            elif isinstance(obj, str):
                add_record(obj)
    else:
        for line in content.splitlines():
            add_record(line)

if shakespeare_path.exists():
    shakespeare_text = shakespeare_path.read_text(encoding='utf-8', errors='ignore')
    shakespeare_lines = [normalize_text(line) for line in shakespeare_text.splitlines()]
    stories.extend([line for line in shakespeare_lines if 20 <= len(line) <= 900])

# Add a small synthetic reasoning / tool-call corpus to anchor the learning objective.
tool_examples = [
    'User: Add 7 and 5. Assistant: 12',
    'User: What color is the sky on a clear day? Assistant: Blue.',
    'User: Use the calculator to compute 9 * 6. Assistant: {"tool": "calculator.run", "arguments": {"expression": "9 * 6"}}',
    'User: Find the weather in Paris. Assistant: {"tool": "weather.get", "arguments": {"city": "Paris"}}',
    'User: Summarize this in one sentence. Assistant: A small child found a warm lantern and followed it into a cozy room.',
]

stories.extend(tool_examples)

clean_text = '\n\n'.join(stories[:30000])
output_path = raw_root / 'clean_corpus.txt'
output_path.write_text(clean_text, encoding='utf-8')

print(f'Collected {len(stories)} cleaned text records.')
print(f'Saved clean corpus to: {output_path}')
print('\nSample excerpt:\n')
print(clean_text[:500])

Collected 23715 cleaned text records.
Saved clean corpus to: /content/raw_dataset/clean_corpus.txt

Sample excerpt:

Before we proceed any further, hear me speak.

You are all resolved rather to die than to famish?

First, you know Caius Marcius is chief enemy to the people.

We know't, we know't.

Let us kill him, and we'll have corn at our own price.

No more talking on't; let it be done: away, away!

One word, good citizens.

We are accounted poor citizens, the patricians good.

What authority surfeits on would relieve us: if they

would yield us but the superfluity, while it were

wholesome, we might guess


### Package the prepared dataset for training
After the text has been cleaned and the tokenized artifacts are written, we move the final training files into a dedicated `dataset` folder so the original raw download can be removed to save space.


## 4. Exploratory Data Analysis (EDA)

This stage helps us understand the quality and shape of the data before we tokenize it.

We inspect:
- total number of characters and lines,
- average line length,
- example sentence structure,
- token density and repetition patterns.

In [22]:
from collections import Counter
import statistics

corpus_path = raw_root / 'clean_corpus.txt'
text = corpus_path.read_text(encoding='utf-8', errors='ignore')
lines = [line for line in text.splitlines() if line.strip()]

lengths = [len(line) for line in lines]
word_counts = [len(line.split()) for line in lines]
char_counts = Counter(text)

print('Corpus path:', corpus_path)
print('Total characters:', len(text))
print('Total lines:', len(lines))
print('Average line length:', round(statistics.mean(lengths), 2))
print('Median line length:', statistics.median(lengths))
print('Average words per line:', round(statistics.mean(word_counts), 2))
print('Top 20 most common characters:', char_counts.most_common(20))

print('\nSample lines:')
for line in lines[:5]:
    print('-', line)


Corpus path: /content/raw_dataset/clean_corpus.txt
Total characters: 1023758
Total lines: 23715
Average line length: 41.17
Median line length: 42
Average words per line: 7.91
Top 20 most common characters: [(' ', 163810), ('e', 91216), ('t', 64899), ('o', 63351), ('a', 53701), ('h', 49992), ('s', 48002), ('\n', 47428), ('n', 46656), ('r', 46542), ('i', 43732), ('l', 32136), ('d', 30180), ('u', 25738), ('m', 21466), ('y', 19708), (',', 19169), ('w', 17062), ('f', 15465), ('c', 15086)]

Sample lines:
- Before we proceed any further, hear me speak.
- You are all resolved rather to die than to famish?
- First, you know Caius Marcius is chief enemy to the people.
- We know't, we know't.
- Let us kill him, and we'll have corn at our own price.


## 5. Feature Engineering

At this step, we convert the text into token IDs so the model can learn from sequences.

The most important choices are:
- vocabulary size,
- tokenizer type,
- context length,
- and training sequence packing.

For a small educational model, a simple byte-level or character-level tokenizer is easiest to debug. A subword BPE tokenizer is more realistic for larger LLMs, but a byte-level tokenizer is a perfect starting point here.

In [24]:
import numpy as np
import tiktoken
from pathlib import Path
import shutil
import json

corpus_path = RAW_ROOT / 'clean_corpus.txt'
text = corpus_path.read_text(encoding='utf-8', errors='ignore')

# Start with a byte-level tokenizer to keep the notebook lightweight and easy to inspect.
enc = tiktoken.get_encoding('gpt2')
encoded = enc.encode(text)

# Truncate to a manageable size for quick experiments
encoded = encoded[:150000]

# Split into train / validation
split = int(0.9 * len(encoded))
train_ids = encoded[:split]
val_ids = encoded[split:]

# Save as binary files for fast loading later
train_path = raw_root / 'train.bin'
val_path = raw_root / 'val.bin'
train_path.write_bytes(np.array(train_ids, dtype=np.uint16).tobytes())
val_path.write_bytes(np.array(val_ids, dtype=np.uint16).tobytes())

print('Encoding length:', len(encoded))
print('Train tokens:', len(train_ids))
print('Validation tokens:', len(val_ids))
print('Saved train.bin and val.bin')

# Package the prepared training artifacts into a dedicated dataset folder.
prepared_dataset_dir = DATASET_ROOT
prepared_dataset_dir.mkdir(parents=True, exist_ok=True)

for asset in [train_path, val_path, corpus_path]:
    if asset.exists():
        shutil.copy2(asset, prepared_dataset_dir / asset.name)

manifest = {
    'train_bin': str(prepared_dataset_dir / 'train.bin'),
    'val_bin': str(prepared_dataset_dir / 'val.bin'),
    'clean_corpus': str(prepared_dataset_dir / 'clean_corpus.txt'),
    'notes': 'Prepared for small GPT-style training in Colab.'
}

manifest_path = prepared_dataset_dir / 'manifest.json'
manifest_path.write_text(json.dumps(manifest, indent=2), encoding='utf-8')

print('Prepared dataset folder created at:', prepared_dataset_dir)
print('Manifest saved at:', manifest_path)

if USE_COLAB:
    drive_dataset_dir = DRIVE_PROJECT_ROOT / 'dataset'
    shutil.copytree(prepared_dataset_dir, drive_dataset_dir, dirs_exist_ok=True)

    print('Copied prepared dataset snapshot to Drive:', drive_dataset_dir)# print('Removed raw_dataset to save disk space.')

# shutil.rmtree(RAW_ROOT, ignore_errors=True)
# Optional cleanup once this folder is confirmed:

Encoding length: 150000
Train tokens: 135000
Validation tokens: 15000
Saved train.bin and val.bin
Prepared dataset folder created at: /content/dataset
Manifest saved at: /content/dataset/manifest.json
Copied prepared dataset snapshot to Drive: /content/drive/MyDrive/GheremiahAI/dataset


## 6. Model Selection

We will use a compact decoder-only Transformer with:
- token embeddings,
- causal self-attention,
- a small feed-forward network,
- and a final language-model head.

This is the same high-level architecture used by modern GPT-style models, but scaled down for learning and experimentation.

Recommended starter configuration:
- vocabulary size: from the tokenizer,
- context length: 256,
- embedding dimension: 256,
- heads: 4,
- layers: 4,
- dropout: 0.1.

In [25]:
import torch
import torch.nn as nn
from torch.nn import functional as F

class CausalSelfAttention(nn.Module):
    def __init__(self, embd_dim, n_head, block_size, dropout=0.1):
        super().__init__()
        assert embd_dim % n_head == 0
        self.c_attn = nn.Linear(embd_dim, 3 * embd_dim)
        self.c_proj = nn.Linear(embd_dim, embd_dim)
        self.n_head = n_head
        self.head_size = embd_dim // n_head
        self.block_size = block_size
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.c_attn(x).split(C, dim=2)
        q = q.view(B, T, self.n_head, self.head_size).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_size).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_size).transpose(1, 2)

        att = (q @ k.transpose(-2, -1)) / (self.head_size ** 0.5)
        mask = torch.tril(torch.ones(T, T, device=x.device)).view(1, 1, T, T)
        att = att.masked_fill(mask == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        att = self.dropout(att)
        y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.c_proj(y)
        return self.dropout(y)

class FeedForward(nn.Module):
    def __init__(self, embd_dim, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(embd_dim, 4 * embd_dim),
            nn.GELU(),
            nn.Linear(4 * embd_dim, embd_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class GPTBlock(nn.Module):
    def __init__(self, embd_dim, n_head, block_size, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(embd_dim)
        self.attn = CausalSelfAttention(embd_dim, n_head, block_size, dropout)
        self.ln2 = nn.LayerNorm(embd_dim)
        self.ff = FeedForward(embd_dim, dropout)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

class TinyGPT(nn.Module):
    def __init__(self, vocab_size, embd_dim=256, n_head=4, n_layer=4, block_size=256, dropout=0.1):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, embd_dim)
        self.position_embedding = nn.Embedding(block_size, embd_dim)
        self.blocks = nn.Sequential(*[GPTBlock(embd_dim, n_head, block_size, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(embd_dim)
        self.head = nn.Linear(embd_dim, vocab_size)

    def forward(self, idx):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device)
        x = self.token_embedding(idx) + self.position_embedding(pos)
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        logits = self.head(x)
        return logits

vocab_size = enc.n_vocab
model = TinyGPT(vocab_size=vocab_size, embd_dim=256, n_head=4, n_layer=4, block_size=256).to(device)
print('Model on device:', next(model.parameters()).device)
print(model)


Model on device: cuda:0
TinyGPT(
  (token_embedding): Embedding(50257, 256)
  (position_embedding): Embedding(256, 256)
  (blocks): Sequential(
    (0): GPTBlock(
      (ln1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (attn): CausalSelfAttention(
        (c_attn): Linear(in_features=256, out_features=768, bias=True)
        (c_proj): Linear(in_features=256, out_features=256, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (ln2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (ff): FeedForward(
        (net): Sequential(
          (0): Linear(in_features=256, out_features=1024, bias=True)
          (1): GELU(approximate='none')
          (2): Linear(in_features=1024, out_features=256, bias=True)
          (3): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (1): GPTBlock(
      (ln1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (attn): CausalSelfAttention(
        (c_attn): Linear(in_features=256, out_fea

## 7. Training & Evaluation

We train the model using teacher-forced next-token prediction. The objective is cross-entropy loss.

The training loop should:
- create batches from `train.bin`,
- compute logits for the next token,
- backpropagate loss,
- periodic validation,
- and save checkpoints.

In [26]:
from torch.utils.data import Dataset, DataLoader

class TextDataset(Dataset):
    def __init__(self, path, block_size=256):
        data = np.fromfile(path, dtype=np.uint16)
        self.data = torch.tensor(data, dtype=torch.long)
        self.block_size = block_size

    def __len__(self):
        return max(1, len(self.data) - self.block_size)

    def __getitem__(self, idx):
        x = self.data[idx:idx + self.block_size]
        y = self.data[idx + 1:idx + 1 + self.block_size]
        return x, y

torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

train_ds = TextDataset(train_path, block_size=256)
val_ds = TextDataset(val_path, block_size=256)

batch_size = 16 if torch.cuda.is_available() else 4
train_loader = DataLoader(
    train_ds,
    batch_size=batch_size,
    shuffle=True,
    pin_memory=torch.cuda.is_available(),
    num_workers=2,
)
val_loader = DataLoader(
    val_ds,
    batch_size=batch_size,
    shuffle=False,
    pin_memory=torch.cuda.is_available(),
    num_workers=2,
)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
criterion = nn.CrossEntropyLoss()

train_losses = []
val_losses = []

for epoch in range(2):
    model.train()
    total_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        logits = model(xb)
        loss = criterion(logits.view(-1, vocab_size), yb.view(-1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg_train = total_loss / max(1, len(train_loader))
    train_losses.append(avg_train)

    model.eval()
    total_val = 0.0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
            logits = model(xb)
            loss = criterion(logits.view(-1, vocab_size), yb.view(-1))
            total_val += loss.item()

    avg_val = total_val / max(1, len(val_loader))
    val_losses.append(avg_val)

    print(f'Epoch {epoch+1}: train_loss={avg_train:.4f}, val_loss={avg_val:.4f}')

print('Training complete.')
print('Train losses:', train_losses)
print('Validation losses:', val_losses)

Epoch 1: train_loss=1.8799, val_loss=8.9322
Epoch 2: train_loss=0.2172, val_loss=10.7638
Training complete.
Train losses: [1.8798622836397991, 0.21722783435802226]
Validation losses: [8.932171316829525, 10.763818981849191]


## 8. Deployment & Monitoring

After training, the model should be evaluated with generation samples and saved for later inference.

We will:
- load the best checkpoint,
- evaluate on short prompts,
- verify output coherence,
- and define a simple deployment path for local inference or export.

In [27]:
@torch.no_grad()
def generate(prompt, max_new_tokens=50):
    model.eval()
    encoded_prompt = enc.encode(prompt)
    input_ids = torch.tensor(encoded_prompt, dtype=torch.long).unsqueeze(0).to(device)

    for _ in range(max_new_tokens):
        if input_ids.shape[1] > 256:
            input_ids = input_ids[:, -256:]
        logits = model(input_ids)[:, -1, :]
        probs = torch.softmax(logits, dim=-1)
        next_idx = torch.multinomial(probs, num_samples=1)
        input_ids = torch.cat([input_ids, next_idx], dim=1)

    generated = enc.decode(input_ids[0].tolist())
    return generated

prompt = 'Once upon a time'
print('Using device for generation:', device)
print(generate(prompt, max_new_tokens=80))

# Save model checkpoint
checkpoint_path = '/content/checkpoint_tinygpt.pt'
torch.save(model.state_dict(), checkpoint_path)
print('Saved checkpoint:', checkpoint_path)

# Save the trained model and dataset using the chosen environment.
from pathlib import Path
import shutil

save_dir = CHECKPOINT_ROOT
save_dir.mkdir(parents=True, exist_ok=True)

if Path(checkpoint_path).exists():
    shutil.copy2(checkpoint_path, save_dir / Path(checkpoint_path).name)
    print('Checkpoint copied to:', save_dir / Path(checkpoint_path).name)
else:
    print('Checkpoint not found yet. Run the training cell first.')

prepared_dataset_dir = DATASET_ROOT
if prepared_dataset_dir.exists():
    target_dataset_dir = save_dir / 'dataset'
    shutil.copytree(prepared_dataset_dir, target_dataset_dir, dirs_exist_ok=True)
    print('Prepared dataset copied to:', target_dataset_dir)
else:
    print('Prepared dataset folder not found yet. Run the dataset packaging cell first.')

Using device for generation: cuda
Once upon a time

For Jesu Christ in glorious Christian field,

Streaming the ensign of the Christian cross

Against black pagans, Turks, and Saracens:

And toil'd with works of war, retired himself

To Italy; and there at Venice gave

His body to that pleasant country's earth,

And his pure soul unto his captain Christ
Saved checkpoint: /content/checkpoint_tinygpt.pt
Checkpoint copied to: /content/drive/MyDrive/GheremiahAI/checkpoints/checkpoint_tinygpt.pt
Prepared dataset copied to: /content/drive/MyDrive/GheremiahAI/checkpoints/dataset


In [29]:
# Save the trained model and dataset to Drive
from pathlib import Path
import shutil

save_dir = Path('/content/drive/GheremiahAI/checkpoints')
save_dir.mkdir(parents=True, exist_ok=True)

checkpoint_path = Path('/content/checkpoint_tinygpt.pt')
if checkpoint_path.exists():
    shutil.copy2(checkpoint_path, save_dir / checkpoint_path.name)
    print('Checkpoint copied to Drive:', save_dir / checkpoint_path.name)
else:
    print('Checkpoint not found yet. Run the training cell first.')

prepared_dataset_dir = Path('/content/dataset')
if prepared_dataset_dir.exists():
    target_dataset_dir = save_dir / 'dataset'
    shutil.copytree(prepared_dataset_dir, target_dataset_dir, dirs_exist_ok=True)
    print('Prepared dataset copied to Drive:', target_dataset_dir)
else:
    print('Prepared dataset folder not found yet. Run the dataset packaging cell first.')

OSError: [Errno 95] Operation not supported: '/content/drive/GheremiahAI'